# FFTW C2C Numba GPU

- https://github.com/cupy


In [41]:
#=-----------------------------------------------------------------------=#

In [1]:
! python --version

Python 3.6.12 :: Anaconda, Inc.


In [2]:
import numpy
numpy.version.version

'1.19.2'

In [2]:
import cupy

In [4]:
# verificando se Cuda está ativo
from numba import cuda
print(cuda.gpus)

<Managed Device 0>, <Managed Device 1>, <Managed Device 2>, <Managed Device 3>


In [1]:
! lscpu

Architecture:          x86_64
CPU op-mode(s):        32-bit, 64-bit
Byte Order:            Little Endian
CPU(s):                88
On-line CPU(s) list:   0-87
Thread(s) per core:    2
Core(s) per socket:    22
Socket(s):             2
NUMA node(s):          2
Vendor ID:             GenuineIntel
CPU family:            6
Model:                 85
Model name:            Intel(R) Xeon(R) Gold 6152 CPU @ 2.10GHz
Stepping:              4
CPU MHz:               2101.000
CPU max MHz:           2101.0000
CPU min MHz:           1000.0000
BogoMIPS:              4200.00
Virtualization:        VT-x
L1d cache:             32K
L1i cache:             32K
L2 cache:              1024K
L3 cache:              30976K
NUMA node0 CPU(s):     0-21,44-65
NUMA node1 CPU(s):     22-43,66-87
Flags:                 fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush dts acpi mmx fxsr sse sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm constant_tsc art arch_perfmon pebs bts rep_good nopl 

In [2]:
! cat /proc/meminfo | grep MemTotal

MemTotal:       791009752 kB


In [3]:
! nvidia-smi

Tue Mar  2 16:01:34 2021       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 418.87.00    Driver Version: 418.87.00    CUDA Version: 10.1     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|===============================+======================+======================|
|   0  Tesla V100-PCIE...  Off  | 00000000:3B:00.0 Off |                    0 |
| N/A   37C    P0    27W / 250W |    108MiB / 32480MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
|   1  Tesla V100-PCIE...  Off  | 00000000:5E:00.0 Off |                    0 |
| N/A   30C    P0    25W / 250W |     15MiB / 32480MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
|   2  T

In [ ]:
import numpy as np, cupy as cp, time as tm
t0 = tm.time()    # <--- time measurement
N = 500
f = np.fromfunction( lambda i, j, k: (i*N**2 + j*N  + k + 1)*1E-6,
                     (N, N, N), dtype=np.complex128 )
ff = cp.fft.fftn( f )
s = cp.sum( ff )
t1 = tm.time()    # <--- time measurement
print('S: {:.2f}    T: {:.4f} s'.format(s, t1-t0))

In [21]:
import cupy as cp
t = cp.linspace(0, 1, 1000)
a = cp.sin(4 * t*2*3.1415)
fft = cp.fft.fft(a)
cp.sum(fft)

array(0.+6.35498598e-14j)

## Run on Sdumont18 (Sequana X)

    ssh sdumont18    # Xeon Gold 6152, 22 cores, 44 threads,
                     # total 88 "cpus", 754 G RAM, 4x Tesla V100
    module load sequana/current
    module load sdbase
    module load anaconda3/2018.12
    conda activate /scratch${PWD#"/prj"}/env2
    export NUMBAPRO_NVVM=/usr/local/cuda-10.1/nvvm/lib64/libnvvm.so
    export NUMBAPRO_LIBDEVICE=/usr/local/cuda-10.1/nvvm/libdevice/

In [46]:
import numpy as np, cupy as cp, time as tm
def f() :
    t0 = tm.time()    # <--- time measurement
    N = 500
    a = np.fromfunction( lambda i, j, k: (i*N**2 + j*N  + k + 1)*1E-6,
                         (N, N, N), dtype=np.complex128 )
    f = cp.asarray(a)
    fft = cp.fft.fftn(f)
    s = cp.sum(fft)
    t1 = tm.time()    # <--- time measurement
    print('S: %.2f%+.2fj    T: %.4f s' % (s.real, s.imag, t1-t0))

In [47]:
f()

S: 125.00-0.00j    T: 4.2147 s


In [48]:
f()

S: 125.00-0.00j    T: 4.1326 s


In [49]:
f()

S: 125.00-0.00j    T: 4.0954 s


## Run on B715

Fila | Wall-clock máximo (em horas) | Número mínimo de nós (núcleos+ dispositivos) | Número máximo de nós (núcleos+ dispositivos) | Número máximo de tarefas em execução por usuário | Número máximo de tarefas em fila por usuário
- | - | - | - | - | -
nvidia_small | 1 | 1 (24+2) | 20 (480+40) | 4 | 24
nvidia_dev | 0:20 | 1 (24+2) | 4 (96+8) | 1 | 1
sequana_gpu | 96 | 1 (48+4) | 21 (1008+84) | 4 | 24

In [1]:
%%writefile gnb715.py
import numpy as np, cupy as cp, time as tm
t0 = tm.time()    # <--- time measurement
N = 500
a = np.fromfunction( lambda i, j, k: (i*N**2 + j*N  + k + 1)*1E-6,
                     (N, N, N), dtype=np.complex128 )
f = cp.asarray(a)
fft = cp.fft.fftn(f)
s = cp.sum(fft)
t1 = tm.time()    # <--- time measurement
print('S: %.2f%+.2fj    T: %.4f s' % (s.real, s.imag, t1-t0))

Writing gnb715.py


In [2]:
! cp gnb715.py /scratch${PWD#"/prj"}

In [10]:
%%writefile gnb715.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#   nvidia_dev: 0:20, 1 (24+2), 4 (96+8), 1, 1
#SBATCH --partition nvidia_dev # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name gnb715      # Job name
#SBATCH --time=00:05:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST
                                              
# Environment
cd
cd /scratch${PWD#"/prj"}/
module load sequana/current
module load sdbase
# module load anaconda3/2020.11
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate ./env2
export NUMBAPRO_NVVM=/usr/local/cuda-10.1/nvvm/lib64/libnvvm.so
export NUMBAPRO_LIBDEVICE=/usr/local/cuda-10.1/nvvm/libdevice/
cd  fft
echo '-- lscpu ------------------------------'
lscpu
echo '-- meminfo -------------------------'
cat /proc/meminfo | grep MemTotal
echo '-- nvidia-smi ----------------------'
nvidia-smi
                                              
# Executable
EXEC="python gnb715.py"

# Start
echo '$ srun  --mpi=pmi2  -n' $SLURM_NTASKS  ${EXEC##*/}
echo '-- output -----------------------------'
srun  --mpi=pmi2  -n $SLURM_NTASKS  $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting gnb715.srm


In [11]:
! sbatch gnb715.srm

Submitted batch job 877996


In [ ]:
! squeue -n gnb715

In [13]:
! squeue -n gnb715

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


In [14]:
! cat /scratch${PWD#"/prj"}/slurm-877996.out

- Job ID: 877996
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont3158
Loading SEQUANA Software environment
Loading SDumont BASE Software environment
-- lscpu ------------------------------
Architecture:          x86_64
CPU op-mode(s):        32-bit, 64-bit
Byte Order:            Little Endian
CPU(s):                24
On-line CPU(s) list:   0-23
Thread(s) per core:    1
Core(s) per socket:    12
Socket(s):             2
NUMA node(s):          2
Vendor ID:             GenuineIntel
CPU family:            6
Model:                 62
Model name:            Intel(R) Xeon(R) CPU E5-2695 v2 @ 2.40GHz
Stepping:              4
CPU MHz:               2399.707
CPU max MHz:           2400.0000
CPU min MHz:           1200.0000
BogoMIPS:              4800.15
Virtualization:        VT-x
L1d cache:             32K
L1i cache:             32K
L2 cache:              256K
L3 cache:              30720K
NUMA n

In [15]:
! sbatch gnb715.srm

Submitted batch job 878017


In [ ]:
! squeue -n gnb715

In [19]:
! squeue -n gnb715

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


In [20]:
! cat /scratch${PWD#"/prj"}/slurm-878017.out

- Job ID: 878017
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont3158
Loading SEQUANA Software environment
Loading SDumont BASE Software environment
-- lscpu ------------------------------
Architecture:          x86_64
CPU op-mode(s):        32-bit, 64-bit
Byte Order:            Little Endian
CPU(s):                24
On-line CPU(s) list:   0-23
Thread(s) per core:    1
Core(s) per socket:    12
Socket(s):             2
NUMA node(s):          2
Vendor ID:             GenuineIntel
CPU family:            6
Model:                 62
Model name:            Intel(R) Xeon(R) CPU E5-2695 v2 @ 2.40GHz
Stepping:              4
CPU MHz:               2400.439
CPU max MHz:           2400.0000
CPU min MHz:           1200.0000
BogoMIPS:              4800.15
Virtualization:        VT-x
L1d cache:             32K
L1i cache:             32K
L2 cache:              256K
L3 cache:              30720K
NUMA n

In [21]:
! sbatch gnb715.srm

Submitted batch job 878019


In [ ]:
! squeue -n gnb715

In [1]:
! squeue -n gnb715

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-878019.out

- Job ID: 878019
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont3158
Loading SEQUANA Software environment
Loading SDumont BASE Software environment
-- lscpu ------------------------------
Architecture:          x86_64
CPU op-mode(s):        32-bit, 64-bit
Byte Order:            Little Endian
CPU(s):                24
On-line CPU(s) list:   0-23
Thread(s) per core:    1
Core(s) per socket:    12
Socket(s):             2
NUMA node(s):          2
Vendor ID:             GenuineIntel
CPU family:            6
Model:                 62
Model name:            Intel(R) Xeon(R) CPU E5-2695 v2 @ 2.40GHz
Stepping:              4
CPU MHz:               2255.566
CPU max MHz:           2400.0000
CPU min MHz:           1200.0000
BogoMIPS:              4800.15
Virtualization:        VT-x
L1d cache:             32K
L1i cache:             32K
L2 cache:              256K
L3 cache:              30720K
NUMA n